In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

df = pd.read_csv("../orders_dataset.csv")

print("Shape:", df.shape)
df.head()

Shape: (6000, 13)


,order_id,product_category,price_inr,discount_pct,payment_method,customer_tenure_days,num_previous_orders,num_previous_returns,delivery_distance_km,delivery_days,is_weekend_order,rating_given,returned
0,1,Footwear,2572.0,23.8,Prepaid_Card,17,3,0,604.6,1,0,2.0,0
1,2,Electronics,16689.0,6.7,COD,104,3,0,166.4,9,0,2.0,0
2,3,Footwear,800.0,51.3,Prepaid_Card,103,3,0,418.8,8,0,3.0,0
3,4,Home,7930.0,49.7,COD,1479,33,4,143.1,5,0,3.0,1
4,5,Apparel,715.0,0.0,COD,95,0,0,335.5,3,1,2.0,0


In [2]:
X = df.drop(columns=["returned"])
y = df["returned"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nTarget distribution:")
print(y.value_counts())
print("\nTarget proportions:")
print(y.value_counts(normalize=True).round(4))

X shape: (6000, 12)
y shape: (6000,)

Target distribution:
returned
0    4635
1    1365
Name: count, dtype: int64

Target proportions:
returned
0    0.7725
1    0.2275
Name: proportion, dtype: float64


In [3]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Numeric features:
['order_id', 'price_inr', 'discount_pct', 'customer_tenure_days', 'num_previous_orders', 'num_previous_returns', 'delivery_distance_km', 'delivery_days', 'is_weekend_order', 'rating_given']

Categorical features:
['product_category', 'payment_method']


In [4]:
X = X.drop(columns=["order_id"])

print("X shape:", X.shape)
print("\nFeatures:")
print(X.columns.tolist())

X shape: (6000, 11)

Features:
['product_category', 'price_inr', 'discount_pct', 'payment_method', 'customer_tenure_days', 'num_previous_orders', 'num_previous_returns', 'delivery_distance_km', 'delivery_days', 'is_weekend_order', 'rating_given']


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True).round(4))

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True).round(4))

Training set: (4800, 11)
Test set: (1200, 11)

Training target distribution:
returned
0    0.7725
1    0.2275
Name: proportion, dtype: float64

Test target distribution:
returned
0    0.7725
1    0.2275
Name: proportion, dtype: float64


In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [7]:
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Numeric features:
['price_inr', 'discount_pct', 'customer_tenure_days', 'num_previous_orders', 'num_previous_returns', 'delivery_distance_km', 'delivery_days', 'is_weekend_order', 'rating_given']

Categorical features:
['product_category', 'payment_method']


In [8]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [9]:
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

In [10]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

In [11]:
preprocessing_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor)
])

In [13]:
print("Preprocessor created successfully.")
print(preprocessor)

Preprocessor created successfully.
ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['price_inr', 'discount_pct',
                                  'customer_tenure_days', 'num_previous_orders',
                                  'num_previous_returns',
                                  'delivery_distance_km', 'delivery_days',
                                  'is_weekend_order', 'rating_given']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
   

In [14]:
from sklearn.dummy import DummyClassifier

In [15]:
dummy_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", DummyClassifier(strategy="most_frequent"))
])

In [22]:
dummy_model.fit(X_train, y_train)

y_pred_dummy = dummy_model.predict(X_test)

In [23]:
from sklearn.metrics import accuracy_score, f1_score

dummy_accuracy = accuracy_score(y_test, y_pred_dummy)
dummy_f1 = f1_score(y_test, y_pred_dummy, pos_label=1)

print("DummyClassifier Accuracy:", round(dummy_accuracy, 4))
print("DummyClassifier F1-score (returned=1):", round(dummy_f1, 4))

DummyClassifier Accuracy: 0.7725
DummyClassifier F1-score (returned=1): 0.0


### DummyClassifier Baseline — Interpretation

The DummyClassifier achieved an accuracy of 77.25%, but its F1-score for the `returned=1` class was 0.0. This happens because the most-frequent strategy predicts every order as `returned=0`, since non-returned orders are the majority class. Although this produces a seemingly high accuracy, the model has zero recall for actual returns and therefore fails to identify any returned orders. This demonstrates the **high accuracy, zero recall** trap: accuracy alone can be misleading when the target classes are imbalanced. Therefore, model performance should be compared against a baseline and evaluated using metrics that are aligned with the real business problem, such as recall, precision, F1-score, and ROC-AUC, rather than relying on accuracy alone.

In [24]:
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        random_state=42,
        max_iter=1000
    ))
])

In [25]:
logistic_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['price_inr', 'discount_pct',
                                                   'customer_tenure_days',
                                                   'num_previous_orders',
                                                   'num_previous_returns',
                                                   'delivery_distance_km',
                                                   'delivery_days',
                                                   'is_weekend_order',
                                                   'rating_given']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['product_category',
                                                   'payment_method'])])),
                ('classifier',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    random_state=42))])

In [26]:
y_pred_log = logistic_model.predict(X_test)
y_proba_log = logistic_model.predict_proba(X_test)[:, 1]

In [27]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    precision_score,
    roc_auc_score
)

log_accuracy = accuracy_score(y_test, y_pred_log)
log_f1 = f1_score(y_test, y_pred_log, pos_label=1)
log_recall = recall_score(y_test, y_pred_log, pos_label=1)
log_precision = precision_score(y_test, y_pred_log, pos_label=1)
log_roc_auc = roc_auc_score(y_test, y_proba_log)

print("Logistic Regression — Default Threshold (0.5)")
print("Accuracy :", round(log_accuracy, 4))
print("F1       :", round(log_f1, 4))
print("Recall   :", round(log_recall, 4))
print("Precision:", round(log_precision, 4))
print("ROC-AUC  :", round(log_roc_auc, 4))

Logistic Regression — Default Threshold (0.5)
Accuracy : 0.5917
F1       : 0.3921
Recall   : 0.5788
Precision: 0.2964
ROC-AUC  : 0.6253


In [28]:
import numpy as np

thresholds = np.arange(0.10, 0.901, 0.01)

In [31]:
threshold_results = []

for threshold in thresholds:
    y_pred_threshold = (y_proba_log >= threshold).astype(int)
    
    f1 = f1_score(y_test, y_pred_threshold, pos_label=1, zero_division=0)
    recall = recall_score(y_test, y_pred_threshold, pos_label=1, zero_division=0)
    precision = precision_score(y_test, y_pred_threshold, pos_label=1, zero_division=0)
    
    threshold_results.append({
        "threshold": threshold,
        "f1": f1,
        "recall": recall,
        "precision": precision
    })

threshold_df = pd.DataFrame(threshold_results)

threshold_df.head()

,threshold,f1,recall,precision
0,0.10,0.370672,1.0,0.2275
1,0.11,0.370672,1.0,0.2275
2,0.12,0.370672,1.0,0.2275
3,0.13,0.370672,1.0,0.2275
4,0.14,0.370672,1.0,0.2275


In [32]:
print("Minimum probability:", y_proba_log.min())
print("Maximum probability:", y_proba_log.max())
print("Mean probability:", y_proba_log.mean())

print("\nProbability percentiles:")
print(np.percentile(y_proba_log, [0, 10, 25, 50, 75, 90, 100]))

Minimum probability: 0.2013711249662326
Maximum probability: 0.8976601857946006
Mean probability: 0.4839872714810472

Probability percentiles:
[0.20137112 0.32149733 0.38643683 0.48095446 0.5759084  0.65648326
 0.89766019]


In [33]:
print(threshold_df.to_string(index=False))

 threshold       f1   recall  precision
      0.10 0.370672 1.000000   0.227500
      0.11 0.370672 1.000000   0.227500
      0.12 0.370672 1.000000   0.227500
      0.13 0.370672 1.000000   0.227500
      0.14 0.370672 1.000000   0.227500
      0.15 0.370672 1.000000   0.227500
      0.16 0.370672 1.000000   0.227500
      0.17 0.370672 1.000000   0.227500
      0.18 0.370672 1.000000   0.227500
      0.19 0.370672 1.000000   0.227500
      0.20 0.370672 1.000000   0.227500
      0.21 0.369565 0.996337   0.226856
      0.22 0.369565 0.996337   0.226856
      0.23 0.370068 0.996337   0.227235
      0.24 0.371585 0.996337   0.228380
      0.25 0.370725 0.992674   0.227923
      0.26 0.370014 0.985348   0.227773
      0.27 0.372576 0.985348   0.229718
      0.28 0.375698 0.985348   0.232097
      0.29 0.377652 0.978022   0.234005
      0.30 0.376167 0.959707   0.233929
      0.31 0.373547 0.941392   0.233001
      0.32 0.373708 0.926740   0.234043
      0.33 0.379154 0.919414   0.238820


In [36]:
# Find optimal threshold based on maximum F1 score
best_idx = threshold_df['f1'].idxmax()
best_threshold = threshold_df.loc[best_idx, 'threshold']
best_f1 = threshold_df.loc[best_idx, 'f1']
best_recall = threshold_df.loc[best_idx, 'recall']
best_precision = threshold_df.loc[best_idx, 'precision']

In [37]:
print("Logistic Regression Threshold Optimization")
print("-------------------------------------------")
print("Default threshold:", 0.50)
print("Default recall:", round(log_recall, 4))
print("Default precision:", round(log_precision, 4))
print("Default F1:", round(log_f1, 4))

print("\nF1-maximising threshold:", round(best_threshold, 2))
print("Best F1:", round(best_f1, 4))
print("Recall at best threshold:", round(best_recall, 4))
print("Precision at best threshold:", round(best_precision, 4))

recall_gain = best_recall - log_recall
precision_change = best_precision - log_precision

print("\nRecall improvement:", round(recall_gain, 4))
print("Precision change:", round(precision_change, 4))

Logistic Regression Threshold Optimization
-------------------------------------------
Default threshold: 0.5
Default recall: 0.5788
Default precision: 0.2964
Default F1: 0.3921

F1-maximising threshold: 0.44
Best F1: 0.4091
Recall at best threshold: 0.7582
Precision at best threshold: 0.2801

Recall improvement: 0.1795
Precision change: -0.0163


Business trade-off: Changing the decision threshold from 0.50 to 0.44 makes the model more aggressive in identifying potentially returned orders. This increases recall from 57.88% to 75.82%, meaning the model catches substantially more actual returns. However, precision decreases from 29.64% to 28.01%, so more non-returned orders are incorrectly flagged as potential returns. In business terms, we are accepting more false positives in exchange for reducing false negatives, because failing to identify an order that will actually be returned can be more costly than unnecessarily flagging an order for additional attention.